In [ ]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

REDACTED_THINKING_TRIGGER = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"

In [ ]:
def chat(messages, system=None, thinking=False, thinking_budget=1024):
    params = {"model": model, "max_tokens": 4096, "messages": messages}
    if thinking:
        params["thinking"] = {"type": "enabled", "budget_tokens": thinking_budget}
    if system:
        params["system"] = system
    return client.messages.create(**params)

In [ ]:
# Test 1: Without thinking
response = chat([{"role": "user", "content": "What is 27 * 453?"}])
for block in response.content:
    print(f"[{block.type}] {block.text[:200] if hasattr(block, 'text') else ''}")

In [ ]:
# Test 2: With thinking
response = chat([{"role": "user", "content": "What is 27 * 453? Show your reasoning."}], thinking=True, thinking_budget=2048)
for block in response.content:
    if block.type == "thinking":
        print(f"[thinking] {block.thinking[:300]}...")
    elif block.type == "text":
        print(f"[text] {block.text[:300]}")
    elif block.type == "redacted_thinking":
        print(f"[redacted] ({len(block.data)} chars encrypted)")

In [ ]:
# Test 3: Redacted thinking trigger
response = chat([{"role": "user", "content": REDACTED_THINKING_TRIGGER}], thinking=True, thinking_budget=2048)
for block in response.content:
    if block.type == "redacted_thinking":
        print(f"[redacted] ({len(block.data)} chars encrypted)")
    elif block.type == "thinking":
        print(f"[thinking] {block.thinking[:200]}...")
    elif block.type == "text":
        print(f"[text] {block.text[:200]}")